# Example 12.5 / 2: Check Cut Validity

Companion notebook for **12.5 Decomposition** in the AMPL lab Documentation, section "Check Cut Validity". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Check Cut Validity

Cut signs depend on the primal subproblem convention and the reported dual convention. Derive each cut from the dual actually represented, then test it on a small extensive-form instance. If the subproblem can be infeasible, the algorithm is incomplete until feasibility cuts, complete recourse, or a valid penalty construction is provided.

The next subproblem writes capacity as the one-variable row `Produce <= capacity_fixed` and is
solved at capacity 0 for the low-demand scenario. HiGHS reports the capacity multiplier on the
variable bound (`Produce.rc`) rather than on the row (`CapacityUse.dual`). A cut built from the
row duals alone claims a recourse of at least 12 even at capacity 8, where the true recourse is 0:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param demand >= 0;                                  # low-demand scenario, units
param capacity_fixed >= 0;                          # capacity fixed by the master
var Produce >= 0;                                   # units made in-house
var Outsource >= 0;                                 # units bought outside
minimize RecourseCost: 3 * Outsource;               # outsourcing costs 3 per unit
subject to Supply: Produce + Outsource >= demand;   # meet demand
subject to CapacityUse: Produce <= capacity_fixed;  # one-variable row: dual lands on Produce.rc

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["demand"] = 4
ampl.param["capacity_fixed"] = 0

In [5]:
# run.py
ampl.option["presolve"] = 0
ampl.solve(solver="highs", verbose=False)
supply, capacity = ampl.con["Supply"].dual(), ampl.con["CapacityUse"].dual()
print("RecourseCost", ampl.obj["RecourseCost"].value(), "Supply.dual", supply,
      "CapacityUse.dual", capacity, "Produce.rc", ampl.var["Produce"].rc())
print("row-dual cut at capacity 8:", supply * 4 + capacity * 8)
ampl.param["capacity_fixed"] = 8
ampl.solve(solver="highs", verbose=False)
print("true recourse at capacity 8:", ampl.obj["RecourseCost"].value())

RecourseCost 12.0 Supply.dual 3.0 CapacityUse.dual 0.0 Produce.rc -3.0
row-dual cut at capacity 8: 12.0
true recourse at capacity 8: 0.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param demand := 4;
param capacity_fixed := 0;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0, presolve 0;
solve;
display RecourseCost, Supply.dual, CapacityUse.dual, Produce.rc;
display Supply.dual * 4 + CapacityUse.dual * 8;
let capacity_fixed := 8;
solve;
display RecourseCost;

RecourseCost = 12
Supply.dual = 3
CapacityUse.dual = 0
Produce.rc = -3

(Supply.dual)*4 + (CapacityUse.dual)*8 = 12

RecourseCost = 0

